In [0]:
# https://www.kaggle.com/datasets/nevildhinoja/e-commerce-sales-prediction-dataset
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema", "bronze")

dbutils.widgets.text("container", "raw")
dbutils.widgets.text("datalake", "adlsrenzocavero")
dbutils.widgets.text("csv", "Ecommerce_Sales_Prediction_Dataset")

dbutils.widgets.text("tabla_bronze", "Ecommerce_Sales_Prediction_Bronze")

catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")

container = dbutils.widgets.get("container")
datalake = dbutils.widgets.get("datalake")
csv = dbutils.widgets.get("csv")

tabla_bronze = dbutils.widgets.get("tabla_bronze")

In [0]:
ruta_csv = f"abfss://{container}@{datalake}.dfs.core.windows.net/{csv}.csv"

schema = StructType([
  StructField("Date", StringType(), True),
  StructField("Product_Category", StringType(), True),
  StructField("Price", DoubleType(), True),
  StructField("Discount", DoubleType(), True),
  StructField("Customer_Segment", StringType(), True),
  StructField("Marketing_Spend", DoubleType(), True),
  StructField("Units_Sold", IntegerType(), True),
]) 

raw_df = ( #Lectura
    spark.read
         .option("header", True)
         .schema(schema)
         .csv(ruta_csv)
)

str_cols = [ #Seleccionar columnas String
    field.name
    for field in raw_df.schema.fields
    if isinstance(field.dataType, StringType)
]

bronze_df = raw_df

for c in str_cols: #Eliminar Espacios
  bronze_df = bronze_df.withColumn(c, trim(col(c)))

bronze_df = bronze_df.withColumn( # Convertir a Date
    "Date",
    to_date(col("Date"), "dd-MM-yyyy")
)

bronze_df = bronze_df.withColumn( # Eliminar %
  "Discount",
  regexp_replace(col("Discount"), "%", "").cast("double")
)

# Validar
bronze_df = bronze_df.filter(
  (col("Price").isNotNull()) &
  (col("Units_Sold").isNotNull()) &
  (col("Price") >= 0)
)


bronze_df = ( # Agregar Timestamp
  bronze_df.withColumn("ingestion_timestamp", current_timestamp()))

# Escritura
bronze_df.write.mode("overwrite").saveAsTable(f"{catalogo}.{esquema}.{tabla_bronze}")

In [0]:
def show(df):
    df.limit(5).display()

show(raw_df)
show(bronze_df)